## Q6 — Own Question: Do Faster Pit Stops Correlate with Better Finishing Positions?

We classify each driver per race as "Faster than avg" or "Slower than avg" relative 
to that race's mean pit stop time. Using within-race relative speed (rather than 
absolute seconds) controls for differences across eras and circuits. We then compare 
average finishing position and podium rate (top 3) between the two groups. Note that 
any correlation found is associative, not causal — faster pit stops may reflect 
better-resourced teams rather than pit stop speed directly causing better results.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ── Research Question ─────────────────────────────────────────────────────────
# Do faster pit stops correlate with better finishing positions?
# We examine whether drivers with below-average pit stop times in a race
# tend to finish higher up the order.

In [0]:
df_pit_stops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True)
df_races     = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv",     header=True)
df_results   = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv",   header=True)
df_drivers   = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv",   header=True)

In [0]:
#Parse duration → seconds
df_pit_stops = df_pit_stops.withColumn(
    "duration_sec",
    F.when(
        F.col("duration").contains(":"),
        F.split(F.col("duration"), ":").getItem(0).cast("double") * 60 +
        F.split(F.col("duration"), ":").getItem(1).cast("double")
    ).otherwise(F.col("duration").cast("double"))
)

In [0]:
#Avg pit stop per driver per race 
df_driver_pit = (
    df_pit_stops
    .groupBy("raceId", "driverId")
    .agg(F.round(F.avg("duration_sec"), 3).alias("avg_pit_sec"),
         F.count("*").alias("n_pit_stops"))
)

In [0]:
df_drivers_clean = df_drivers.select(
    "driverId",
    F.concat_ws(" ", F.col("forename"), F.col("surname")).alias("driver_name"),
    F.when(
        F.col("code").isNull() |
        (F.trim(F.col("code")) == "") |
        (F.col("code") == "\\N"),
        F.upper(F.substring(F.col("surname"), 1, 3))
    ).otherwise(F.col("code")).alias("driver_code")
)

In [0]:
df_results_clean = df_results.select(
    "raceId", "driverId",
    F.col("positionOrder").cast("int").alias("finishing_position")
)

In [0]:
df_joined = (
    df_driver_pit
    .join(df_results_clean, on=["raceId", "driverId"], how="left")
    .join(df_races.select("raceId", "year", F.col("name").alias("race_name")),
          on="raceId", how="left")
    .join(df_drivers_clean, on="driverId", how="left")
)

In [0]:
window_race = Window.partitionBy("raceId")

df_joined = (
    df_joined
    .withColumn("race_avg_pit_sec", F.round(F.avg("avg_pit_sec").over(window_race), 3))
    .withColumn("pit_vs_field",
        F.when(F.col("avg_pit_sec") < F.col("race_avg_pit_sec"), "Faster than avg")
         .otherwise("Slower than avg")
    )
)

In [0]:
#Podium flag (top 3)
df_joined = df_joined.withColumn(
    "podium", (F.col("finishing_position") <= 3).cast("int")
)

In [0]:
df_summary = (
    df_joined
    .groupBy("pit_vs_field")
    .agg(
        F.count("*").alias("total_races"),
        F.sum("podium").alias("total_podiums"),
        F.round(F.avg("finishing_position"), 2).alias("avg_finishing_position"),
        F.round(F.avg("avg_pit_sec"), 3).alias("avg_pit_stop_sec"),
    )
    .withColumn(
        "podium_rate_%",
        F.round(F.col("total_podiums") / F.col("total_races") * 100, 2)
    )
    .orderBy("avg_finishing_position")
)

In [0]:
df_q6_detail = (
    df_joined
    .select("year", "race_name", "driver_code", "driver_name",
            "n_pit_stops", "avg_pit_sec", "race_avg_pit_sec",
            "pit_vs_field", "finishing_position", "podium")
    .orderBy("year", "race_name", "finishing_position")
)

In [0]:
print("=== Summary: Pit Stop Speed vs Race Outcome ===")
display(df_summary)

print("=== Detail Table ===")
display(df_q6_detail)